In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
Firms = "/content/drive/MyDrive/India_Firms"
os.listdir(Firms)


['fire_archive_SV-C2_800831.csv',
 'fire_nrt_J2V-C2_810478.csv',
 'fire_archive_J1V-C2_810477.csv',
 'fire_nrt_J1V-C2_810477.csv',
 'India_Firms.parquet']

In [3]:
import pandas as pd

In [4]:
df = pd.read_parquet(f"{Firms}/India_Firms.parquet")

In [5]:
df.head(10)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,frp,daynight,datetime,date,x,y,grid_x,grid_y,time_seconds,time_bin
0,21.95337,87.52301,334.68,0.42,0.61,2024-01-17,635,N21,VIIRS,n,...,7.20,D,2024-01-17 06:35:00,2024-01-17,8.444770e+06,2.734343e+06,8444,2734,1705473300,39478
1,22.13076,88.77950,328.67,0.35,0.57,2024-01-17,635,N21,VIIRS,n,...,2.18,D,2024-01-17 06:35:00,2024-01-17,8.566004e+06,2.755359e+06,8566,2755,1705473300,39478
2,21.95212,87.52202,341.53,0.42,0.61,2024-01-17,635,N21,VIIRS,n,...,7.36,D,2024-01-17 06:35:00,2024-01-17,8.444674e+06,2.734195e+06,8444,2734,1705473300,39478
3,23.81528,92.59270,332.27,0.57,0.43,2024-01-17,635,N21,VIIRS,n,...,5.13,D,2024-01-17 06:35:00,2024-01-17,8.933925e+06,2.953612e+06,8933,2953,1705473300,39478
4,22.41985,88.27590,326.11,0.37,0.58,2024-01-17,635,N21,VIIRS,n,...,9.03,D,2024-01-17 06:35:00,2024-01-17,8.517413e+06,2.789552e+06,8517,2789,1705473300,39478
5,22.42062,88.27940,367.00,0.37,0.58,2024-01-17,635,N21,VIIRS,h,...,13.32,D,2024-01-17 06:35:00,2024-01-17,8.517751e+06,2.789643e+06,8517,2789,1705473300,39478
6,24.82394,94.22270,330.33,0.46,0.39,2024-01-17,637,N21,VIIRS,n,...,4.67,D,2024-01-17 06:37:00,2024-01-17,9.091198e+06,3.071136e+06,9091,3071,1705473420,39478
7,24.45433,93.96598,333.09,0.47,0.40,2024-01-17,637,N21,VIIRS,n,...,5.19,D,2024-01-17 06:37:00,2024-01-17,9.066428e+06,3.028177e+06,9066,3028,1705473420,39478
8,26.59586,95.01118,337.18,0.41,0.37,2024-01-17,637,N21,VIIRS,n,...,8.28,D,2024-01-17 06:37:00,2024-01-17,9.167275e+06,3.275314e+06,9167,3275,1705473420,39478
9,25.36554,93.55044,331.27,0.48,0.40,2024-01-17,637,N21,VIIRS,n,...,18.55,D,2024-01-17 06:37:00,2024-01-17,9.026334e+06,3.133859e+06,9026,3133,1705473420,39478


In [7]:
test_date = "2021-03-31"

day_df = df[
    df["date"].astype(str) == test_date
].copy()

print("Date:", test_date)
print("Number of detections:", len(day_df))

Date: 2021-03-31
Number of detections: 37801


In [8]:
cluster_df = day_df[
    [
        "latitude",
        "longitude",
        "x",
        "y",
        "datetime",
        "satellite",
        "frp",
        "brightness",
        "grid_x",
        "grid_y"
    ]
].copy()

print(cluster_df.shape)

(37801, 10)


In [9]:
from scipy.spatial import cKDTree

coords = cluster_df[
    ["x", "y"]
].to_numpy()

tree = cKDTree(coords)

In [10]:
pairs = tree.query_pairs(
    r=1000,
    output_type="ndarray"
)

print("Spatial candidate pairs:", len(pairs))

Spatial candidate pairs: 89322


In [12]:
import numpy as np

In [13]:
time_seconds = (
    cluster_df["datetime"]
    .astype("int64")
    .to_numpy()
) // 10**9

time_difference = np.abs(
    time_seconds[pairs[:, 0]]
    - time_seconds[pairs[:, 1]]
)

valid_pairs = pairs[
    time_difference <= 12 * 60 * 60
]

print(
    "Pairs satisfying 1 km + 12 hour:",
    len(valid_pairs)
)

Pairs satisfying 1 km + 12 hour: 75626


In [14]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

n = len(cluster_df)

row = np.concatenate([
    valid_pairs[:, 0],
    valid_pairs[:, 1]
])

col = np.concatenate([
    valid_pairs[:, 1],
    valid_pairs[:, 0]
])

data = np.ones(
    len(row),
    dtype=np.int8
)

graph = coo_matrix(
    (data, (row, col)),
    shape=(n, n)
).tocsr()

n_clusters, labels = connected_components(
    graph,
    directed=False
)

cluster_df["cluster_id"] = labels

print("Number of clusters:", n_clusters)

Number of clusters: 12641


In [15]:
print(
    cluster_df[
        [
            "latitude",
            "longitude",
            "datetime",
            "satellite",
            "frp",
            "cluster_id"
        ]
    ].head(20)
)

         latitude  longitude            datetime satellite    frp  cluster_id
3392590  10.33731   77.60508 2021-03-31 07:12:00       N20  24.41           0
3392591  10.33381   77.60721 2021-03-31 07:12:00       N20  17.93           0
3392592  10.30264   77.59853 2021-03-31 07:12:00       N20   8.12           1
3392593   9.73625   77.97352 2021-03-31 07:12:00       N20   5.38           2
3392594  11.26771   78.99136 2021-03-31 07:12:00       N20   3.57           3
3392595  10.29391   77.59176 2021-03-31 07:12:00       N20  16.26           1
3392596  10.05220   78.48373 2021-03-31 07:12:00       N20   5.83           4
3392597  11.26764   78.98911 2021-03-31 07:12:00       N20   4.31           3
3392598   9.74323   77.97209 2021-03-31 07:12:00       N20   5.38           2
3392599  14.83879   79.11230 2021-03-31 07:13:00       N20   6.16           5
3392600  14.86409   79.32521 2021-03-31 07:13:00       N20   6.77           6
3392601  14.76269   79.17649 2021-03-31 07:13:00       N20   5.8

In [16]:
cluster_sizes = (
    cluster_df["cluster_id"]
    .value_counts()
)

print(cluster_sizes.head(20))

cluster_id
3708    95
1257    91
3530    81
6824    81
1329    80
2918    80
2631    77
1293    73
1129    67
885     65
4617    61
1926    61
4408    59
3647    57
4611    56
849     52
587     52
272     50
503     49
2965    48
Name: count, dtype: int64


In [17]:
cluster_summary = (
    cluster_df
    .groupby("cluster_id")
    .agg(
        start_time=("datetime", "min"),
        end_time=("datetime", "max"),
        n_detections=("datetime", "size"),
        centroid_x=("x", "mean"),
        centroid_y=("y", "mean")
    )
    .reset_index()
)

print(cluster_summary.head())

   cluster_id          start_time            end_time  n_detections  \
0           0 2021-03-31 07:12:00 2021-03-31 08:02:00             3   
1           1 2021-03-31 07:12:00 2021-03-31 20:37:00            23   
2           2 2021-03-31 07:12:00 2021-03-31 08:02:00             4   
3           3 2021-03-31 07:12:00 2021-03-31 08:53:00             4   
4           4 2021-03-31 07:12:00 2021-03-31 07:12:00             1   

     centroid_x    centroid_y  
0  7.487883e+06  1.311624e+06  
1  7.486238e+06  1.306968e+06  
2  7.523222e+06  1.236856e+06  
3  7.621429e+06  1.428727e+06  
4  7.572603e+06  1.275997e+06  


In [18]:
cluster_summary["duration_hours"] = (
    cluster_summary["end_time"]
    - cluster_summary["start_time"]
).dt.total_seconds() / 3600

In [19]:
print(
    cluster_summary[
        [
            "cluster_id",
            "start_time",
            "end_time",
            "duration_hours",
            "n_detections"
        ]
    ].head(20)
)

    cluster_id          start_time            end_time  duration_hours  \
0            0 2021-03-31 07:12:00 2021-03-31 08:02:00        0.833333   
1            1 2021-03-31 07:12:00 2021-03-31 20:37:00       13.416667   
2            2 2021-03-31 07:12:00 2021-03-31 08:02:00        0.833333   
3            3 2021-03-31 07:12:00 2021-03-31 08:53:00        1.683333   
4            4 2021-03-31 07:12:00 2021-03-31 07:12:00        0.000000   
5            5 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
6            6 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
7            7 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
8            8 2021-03-31 07:13:00 2021-03-31 07:13:00        0.000000   
9            9 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
10          10 2021-03-31 07:13:00 2021-03-31 07:13:00        0.000000   
11          11 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
12          12 2021-03-31 07:13:00 202

In [20]:
from pyproj import Transformer

centroid_transformer = Transformer.from_crs(
    "EPSG:6933",
    "EPSG:4326",
    always_xy=True
)

centroid_lon, centroid_lat = centroid_transformer.transform(
    cluster_summary["centroid_x"].to_numpy(),
    cluster_summary["centroid_y"].to_numpy()
)

cluster_summary["centroid_lon"] = centroid_lon
cluster_summary["centroid_lat"] = centroid_lat

In [21]:
satellite_summary = (
    cluster_df
    .groupby("cluster_id")["satellite"]
    .agg(
        lambda x: ",".join(
            sorted(x.dropna().astype(str).unique())
        )
    )
    .reset_index(name="satellites")
)

cluster_summary = cluster_summary.merge(
    satellite_summary,
    on="cluster_id",
    how="left"
)

In [22]:
satellite_count = (
    cluster_df
    .groupby("cluster_id")["satellite"]
    .nunique()
    .reset_index(name="n_satellites")
)

cluster_summary = cluster_summary.merge(
    satellite_count,
    on="cluster_id",
    how="left"
)

In [23]:
frp_summary = (
    cluster_df
    .groupby("cluster_id")["frp"]
    .agg(
        total_frp="sum",
        mean_frp="mean",
        max_frp="max"
    )
    .reset_index()
)

cluster_summary = cluster_summary.merge(
    frp_summary,
    on="cluster_id",
    how="left"
)

In [24]:
centroid_x_map = cluster_summary.set_index(
    "cluster_id"
)["centroid_x"]

centroid_y_map = cluster_summary.set_index(
    "cluster_id"
)["centroid_y"]

cluster_df["centroid_x"] = cluster_df["cluster_id"].map(
    centroid_x_map
)

cluster_df["centroid_y"] = cluster_df["cluster_id"].map(
    centroid_y_map
)

In [25]:
cluster_df["distance_from_centroid_m"] = np.sqrt(
    (
        cluster_df["x"]
        - cluster_df["centroid_x"]
    ) ** 2
    +
    (
        cluster_df["y"]
        - cluster_df["centroid_y"]
    ) ** 2
)

In [26]:
extent_summary = (
    cluster_df
    .groupby("cluster_id")["distance_from_centroid_m"]
    .max()
    .reset_index(
        name="max_radius_m"
    )
)

cluster_summary = cluster_summary.merge(
    extent_summary,
    on="cluster_id",
    how="left"
)

In [27]:
cluster_summary["max_radius_km"] = (
    cluster_summary["max_radius_m"] / 1000
)

In [28]:
print(
    cluster_summary[
        [
            "cluster_id",
            "start_time",
            "end_time",
            "duration_hours",
            "centroid_lat",
            "centroid_lon",
            "max_radius_km",
            "n_detections",
            "satellites",
            "n_satellites",
            "total_frp",
            "mean_frp",
            "max_frp"
        ]
    ].head(20)
)

    cluster_id          start_time            end_time  duration_hours  \
0            0 2021-03-31 07:12:00 2021-03-31 08:02:00        0.833333   
1            1 2021-03-31 07:12:00 2021-03-31 20:37:00       13.416667   
2            2 2021-03-31 07:12:00 2021-03-31 08:02:00        0.833333   
3            3 2021-03-31 07:12:00 2021-03-31 08:53:00        1.683333   
4            4 2021-03-31 07:12:00 2021-03-31 07:12:00        0.000000   
5            5 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
6            6 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
7            7 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
8            8 2021-03-31 07:13:00 2021-03-31 07:13:00        0.000000   
9            9 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
10          10 2021-03-31 07:13:00 2021-03-31 07:13:00        0.000000   
11          11 2021-03-31 07:13:00 2021-03-31 08:03:00        0.833333   
12          12 2021-03-31 07:13:00 202

In [29]:
print(
    cluster_summary
    .sort_values(
        "n_detections",
        ascending=False
    )
    .head(20)
)

      cluster_id          start_time            end_time  n_detections  \
3708        3708 2021-03-31 07:16:00 2021-03-31 20:33:00            95   
1257        1257 2021-03-31 07:15:00 2021-03-31 20:34:00            91   
3530        3530 2021-03-31 07:16:00 2021-03-31 20:33:00            81   
6824        6824 2021-03-31 08:05:00 2021-03-31 21:24:00            81   
1329        1329 2021-03-31 07:15:00 2021-03-31 08:04:00            80   
2918        2918 2021-03-31 07:15:00 2021-03-31 21:25:00            80   
2631        2631 2021-03-31 07:15:00 2021-03-31 20:34:00            77   
1293        1293 2021-03-31 07:15:00 2021-03-31 20:34:00            73   
1129        1129 2021-03-31 07:15:00 2021-03-31 20:34:00            67   
885          885 2021-03-31 07:14:00 2021-03-31 20:34:00            65   
4617        4617 2021-03-31 07:16:00 2021-03-31 20:33:00            61   
1926        1926 2021-03-31 07:15:00 2021-03-31 20:34:00            61   
4408        4408 2021-03-31 07:16:00 2

In [30]:
persistence_summary = (
    cluster_df
    .groupby("cluster_id")["datetime"]
    .nunique()
    .reset_index(
        name="n_observation_times"
    )
)

cluster_summary = cluster_summary.merge(
    persistence_summary,
    on="cluster_id",
    how="left"
)

In [31]:
print(cluster_summary.shape)
print(cluster_summary.head())

(12641, 17)
   cluster_id          start_time            end_time  n_detections  \
0           0 2021-03-31 07:12:00 2021-03-31 08:02:00             3   
1           1 2021-03-31 07:12:00 2021-03-31 20:37:00            23   
2           2 2021-03-31 07:12:00 2021-03-31 08:02:00             4   
3           3 2021-03-31 07:12:00 2021-03-31 08:53:00             4   
4           4 2021-03-31 07:12:00 2021-03-31 07:12:00             1   

     centroid_x    centroid_y  duration_hours  centroid_lon  centroid_lat  \
0  7.487883e+06  1.311624e+06        0.833333     77.605677     10.335827   
1  7.486238e+06  1.306968e+06       13.416667     77.588626     10.298747   
2  7.523222e+06  1.236856e+06        0.833333     77.971940      9.740880   
3  7.621429e+06  1.428727e+06        1.683333     78.989767     11.269882   
4  7.572603e+06  1.275997e+06        0.000000     78.483730     10.052200   

  satellites  n_satellites  total_frp   mean_frp  max_frp  max_radius_m  \
0   N20,SNPP           

In [32]:
cluster_summary = cluster_summary.merge(
    persistence_summary,
    on="cluster_id",
    how="left"
)

print(cluster_summary.shape)
print(cluster_summary.head())

(12641, 18)
   cluster_id          start_time            end_time  n_detections  \
0           0 2021-03-31 07:12:00 2021-03-31 08:02:00             3   
1           1 2021-03-31 07:12:00 2021-03-31 20:37:00            23   
2           2 2021-03-31 07:12:00 2021-03-31 08:02:00             4   
3           3 2021-03-31 07:12:00 2021-03-31 08:53:00             4   
4           4 2021-03-31 07:12:00 2021-03-31 07:12:00             1   

     centroid_x    centroid_y  duration_hours  centroid_lon  centroid_lat  \
0  7.487883e+06  1.311624e+06        0.833333     77.605677     10.335827   
1  7.486238e+06  1.306968e+06       13.416667     77.588626     10.298747   
2  7.523222e+06  1.236856e+06        0.833333     77.971940      9.740880   
3  7.621429e+06  1.428727e+06        1.683333     78.989767     11.269882   
4  7.572603e+06  1.275997e+06        0.000000     78.483730     10.052200   

  satellites  n_satellites  total_frp   mean_frp  max_frp  max_radius_m  \
0   N20,SNPP           

In [34]:
print(cluster_summary["n_detections"].describe())
print(cluster_summary["duration_hours"].describe())
print(cluster_summary["max_radius_km"].describe())
print(cluster_summary["n_observation_times_x"].describe())
print(cluster_summary["n_satellites"].value_counts().sort_index())

count    12641.000000
mean         2.990349
std          4.714045
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         95.000000
Name: n_detections, dtype: float64
count    12641.000000
mean         1.249372
std          3.412725
min          0.000000
25%          0.000000
50%          0.000000
75%          0.833333
max         14.166667
Name: duration_hours, dtype: float64
count    12641.000000
mean         0.235812
std          0.394293
min          0.000000
25%          0.000000
50%          0.032303
75%          0.325201
max          5.440171
Name: max_radius_km, dtype: float64
count    12641.000000
mean         1.486829
std          0.782728
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: n_observation_times_x, dtype: float64
n_satellites
1    8172
2    4469
Name: count, dtype: int64


In [35]:
print(
    cluster_df[
        cluster_df["cluster_id"] ==
        cluster_df["cluster_id"].value_counts().index[0]
    ][
        [
            "latitude",
            "longitude",
            "datetime",
            "satellite",
            "frp",
            "cluster_id"
        ]
    ].sort_values("datetime")
)

          latitude  longitude            datetime satellite    frp  cluster_id
3399744   24.96317   81.23261 2021-03-31 07:16:00       N20  16.52        3708
3399788   24.92976   81.29863 2021-03-31 07:16:00       N20   6.15        3708
3399785   24.92657   81.29683 2021-03-31 07:16:00       N20   5.02        3708
3399783   24.93183   81.30053 2021-03-31 07:16:00       N20   5.67        3708
3399782   24.93093   81.29610 2021-03-31 07:16:00       N20   5.02        3708
...            ...        ...                 ...       ...    ...         ...
12227476  24.94620   81.27462 2021-03-31 20:33:00      SNPP   3.12        3708
12227472  24.94152   81.25755 2021-03-31 20:33:00      SNPP   2.22        3708
12227503  24.95996   81.26007 2021-03-31 20:33:00      SNPP   3.45        3708
12227490  24.95336   81.27422 2021-03-31 20:33:00      SNPP   3.97        3708
12227504  24.95900   81.26512 2021-03-31 20:33:00      SNPP   4.67        3708

[95 rows x 6 columns]


In [36]:
df.shape

(15392720, 22)

In [37]:
df.sample(20)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,frp,daynight,datetime,date,x,y,grid_x,grid_y,time_seconds,time_bin
6725157,30.17686,78.84866,338.54,0.51,0.66,2026-05-21,842,N20,VIIRS,n,...,8.49,D,2026-05-21 08:42:00,2026-05-21,7.607814e+06,3.678378e+06,7607,3678,1779352920,41188
4700102,15.27285,79.09679,341.66,0.40,0.37,2023-04-14,804,N20,VIIRS,n,...,3.32,D,2023-04-14 08:04:00,2023-04-14,7.631755e+06,1.926012e+06,7631,1926,1681459440,38922
200772,18.75443,80.81301,330.24,0.41,0.45,2024-04-14,726,N21,VIIRS,l,...,3.51,D,2024-04-14 07:26:00,2024-04-14,7.797347e+06,2.351153e+06,7797,2351,1713079560,39654
2603761,18.81024,81.85027,333.24,0.47,0.39,2020-03-15,759,N20,VIIRS,l,...,1.71,D,2020-03-15 07:59:00,2020-03-15,7.897428e+06,2.357903e+06,7897,2357,1584259140,36672
10666722,15.17549,76.65585,315.38,0.39,0.36,2018-06-06,2030,SNPP,SNPP,n,...,0.93,N,2018-06-06 20:30:00,2018-06-06,7.396238e+06,1.914016e+06,7396,1914,1528317000,35377
8941215,22.36740,86.10107,339.81,0.38,0.43,2016-02-15,748,SNPP,SNPP,n,...,2.78,D,2016-02-15 07:48:00,2016-02-15,8.307572e+06,2.783354e+06,8307,2783,1455522480,33692
1386260,23.66349,87.15774,305.38,0.38,0.59,2026-04-01,1913,N21,VIIRS,n,...,1.42,N,2026-04-01 19:13:00,2026-04-01,8.409526e+06,2.935847e+06,8409,2935,1775070780,41089
12315416,20.23973,78.19186,340.25,0.45,0.63,2021-04-08,714,SNPP,SNPP,n,...,2.18,D,2021-04-08 07:14:00,2021-04-08,7.544442e+06,2.530034e+06,7544,2530,1617866040,37450
10243028,15.17893,77.10495,308.46,0.50,0.49,2018-02-21,1958,SNPP,SNPP,n,...,0.85,N,2018-02-21 19:58:00,2018-02-21,7.439570e+06,1.914440e+06,7439,1914,1519243080,35167
14182228,30.09365,78.70787,333.84,0.41,0.37,2024-05-15,2039,SNPP,SNPP,n,...,3.51,N,2024-05-15 20:39:00,2024-05-15,7.594230e+06,3.669166e+06,7594,3669,1715805540,39717


In [38]:
df["date"] = pd.to_datetime(
    df["date"]
)

In [40]:
unique_dates = (
    df["date"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

print("Number of dates:", len(unique_dates))
print("First:", unique_dates[0])
print("Last:", unique_dates[-1])

Number of dates: 5337
First: 2012-01-20T00:00:00.000000000
Last: 2026-08-31T00:00:00.000000000


In [41]:
date_df = pd.DataFrame({
    "date": unique_dates
})

date_df["previous_date"] = (
    date_df["date"] - pd.Timedelta(days=1)
)

date_df["next_date"] = (
    date_df["date"] + pd.Timedelta(days=1)
)

print(date_df.head())

        date previous_date  next_date
0 2012-01-20    2012-01-19 2012-01-21
1 2012-01-21    2012-01-20 2012-01-22
2 2012-01-22    2012-01-21 2012-01-23
3 2012-01-23    2012-01-22 2012-01-24
4 2012-01-24    2012-01-23 2012-01-25


In [48]:
date1 = pd.Timestamp("2021-03-31")
date2 = pd.Timestamp("2021-04-01")

boundary_df = df[
    df["date"].isin([date1, date2])
][[
    "date",
    "latitude",
    "longitude",
    "x",
    "y",
    "datetime",
    "satellite",
    "frp"
]].copy()

print("Boundary records:", len(boundary_df))
print(boundary_df["date"].value_counts())


Boundary records: 74160
date
2021-03-31    37801
2021-04-01    36359
Name: count, dtype: int64


In [50]:
from scipy.spatial import cKDTree

tree2 = cKDTree(
    df2[["x", "y"]].to_numpy()
)

neighbors = tree2.query_ball_point(
    df1[["x", "y"]].to_numpy(),
    r=1000
)

t1 = (
    df1["datetime"].astype("int64").to_numpy()
) // 10**9

t2 = (
    df2["datetime"].astype("int64").to_numpy()
) // 10**9

cross_pairs = []

for i, neighbor_indices in enumerate(neighbors):
    for j in neighbor_indices:
        if abs(t1[i] - t2[j]) <= 12 * 3600:
            cross_pairs.append((i, j))

print("Valid cross-day pairs:", len(cross_pairs))


Valid cross-day pairs: 15887


In [51]:
df1 = boundary_df[
    boundary_df["date"] == date1
].copy()

df2 = boundary_df[
    boundary_df["date"] == date2
].copy()

print("Day 1:", len(df1))
print("Day 2:", len(df2))


Day 1: 37801
Day 2: 36359


In [53]:
print("First date:", df["date"].min())
print("Last date:", df["date"].max())

print(
    "Number of unique dates:",
    df["date"].nunique()
)


First date: 2012-01-20 00:00:00
Last date: 2026-08-31 00:00:00
Number of unique dates: 5337


In [55]:
all_dates = pd.date_range(
    df["date"].min(),
    df["date"].max(),
    freq="D"
)

observed_dates = pd.DatetimeIndex(
    df["date"].dropna().unique()
)

missing_dates = all_dates.difference(observed_dates)

print("Expected calendar days:", len(all_dates))
print("Observed dates:", len(observed_dates))
print("Missing dates:", len(missing_dates))

print("First missing dates:")
print(missing_dates[:20])


Expected calendar days: 5338
Observed dates: 5337
Missing dates: 1
First missing dates:
DatetimeIndex(['2012-03-25'], dtype='datetime64[ns]', freq='D')


In [57]:
date2 = pd.Timestamp("2021-04-01")

day2_df = df[
    df["date"] == date2
].copy()

print("April 1 detections:", len(day2_df))


April 1 detections: 36359


In [58]:
cluster_df2 = day2_df[
    [
        "datetime",
        "latitude",
        "longitude",
        "x",
        "y",
        "satellite",
        "frp"
    ]
].copy()

cluster_df2 = cluster_df2.reset_index(drop=True)


In [59]:
from scipy.spatial import cKDTree
import numpy as np

coords2 = cluster_df2[
    ["x", "y"]
].to_numpy()

tree2 = cKDTree(coords2)

pairs2 = tree2.query_pairs(
    r=1000,
    output_type="ndarray"
)

print(
    "April 1 spatial candidate pairs:",
    len(pairs2)
)


April 1 spatial candidate pairs: 90202


In [60]:
t2 = (
    cluster_df2["datetime"]
    .astype("int64")
    .to_numpy()
) // 10**9

time_diff2 = np.abs(
    t2[pairs2[:, 0]]
    - t2[pairs2[:, 1]]
)

valid_mask2 = (
    time_diff2 <= 12 * 3600
)

valid_pairs2 = pairs2[valid_mask2]

print(
    "April 1 valid spatio-temporal pairs:",
    len(valid_pairs2)
)


April 1 valid spatio-temporal pairs: 74095


In [61]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

n2 = len(cluster_df2)

rows2 = valid_pairs2[:, 0]
cols2 = valid_pairs2[:, 1]

graph2 = coo_matrix(
    (
        np.ones(len(valid_pairs2), dtype=np.int8),
        (rows2, cols2)
    ),
    shape=(n2, n2)
)

graph2 = graph2 + graph2.T

n_clusters2, labels2 = connected_components(
    graph2,
    directed=False
)

cluster_df2["cluster_id"] = labels2

print(
    "April 1 clusters:",
    n_clusters2
)


April 1 clusters: 12901


In [63]:
day1_boundary = df[
    df["date"] == date1
][
    [
        "datetime",
        "latitude",
        "longitude",
        "x",
        "y",
        "satellite",
        "frp"
    ]
].copy().reset_index(drop=True)


In [65]:
day2_boundary = df[
    df["date"] == date2
][
    [
        "datetime",
        "latitude",
        "longitude",
        "x",
        "y",
        "satellite",
        "frp"
    ]
].copy().reset_index(drop=True)


In [66]:
cluster_df["cluster_id"]


,cluster_id
3392590,0
3392591,0
3392592,1
3392593,2
3392594,3
...,...
12233754,12638
12233755,12638
12233756,8150
12233757,12639


In [67]:
cluster_df2["cluster_id"]


,cluster_id
0,0
1,1
2,2
3,3
4,4
...,...
36354,5450
36355,12900
36356,3372
36357,5294


In [68]:
cross_cluster_pairs = set()

for i, j in cross_pairs:
    c1 = int(cluster_df.loc[i, "cluster_id"])
    c2 = int(cluster_df2.loc[j, "cluster_id"])

    cross_cluster_pairs.add(
        (c1, c2)
    )

print(
    "Unique cross-day cluster connections:",
    len(cross_cluster_pairs)
)


KeyError: 10837

Fixing Of label errors caused by 66 and 67 cell in the 68th cell


In [69]:
# March 31 cluster labels in positional order
march31_labels = (
    cluster_df["cluster_id"]
    .to_numpy()
)

# April 1 cluster labels in positional order
april1_labels = (
    cluster_df2["cluster_id"]
    .to_numpy()
)

print("March labels:", len(march31_labels))
print("April labels:", len(april1_labels))


March labels: 37801
April labels: 36359


In [70]:
cross_cluster_pairs = set()

for i, j in cross_pairs:

    c1 = int(march31_labels[i])
    c2 = int(april1_labels[j])

    cross_cluster_pairs.add(
        (c1, c2)
    )

print(
    "Unique cross-day cluster connections:",
    len(cross_cluster_pairs)
)


Unique cross-day cluster connections: 2066


In [71]:
assert len(march31_labels) == len(df1)
assert len(april1_labels) == len(df2)

print("Index alignment verified.")


Index alignment verified.


In [72]:
print(
    "Detection-level cross-day pairs:",
    len(cross_pairs)
)

print(
    "Unique cross-day cluster connections:",
    len(cross_cluster_pairs)
)

print(
    "Example connections:"
)

print(
    list(cross_cluster_pairs)[:20]
)


Detection-level cross-day pairs: 15887
Unique cross-day cluster connections: 2066
Example connections:
[(883, 7406), (4514, 4170), (89, 77), (7669, 5843), (1050, 84), (7851, 4688), (7658, 92), (178, 5753), (8240, 3974), (7997, 52), (12067, 384), (7238, 3526), (7952, 7219), (422, 6909), (8024, 5783), (3861, 1696), (11482, 3113), (11660, 9572), (234, 5662), (7545, 2672)]


In [73]:
df.shape

(15392720, 22)

In [74]:
cluster_df["local_cluster_id"] = (
    "20210331_" +
    cluster_df["cluster_id"].astype(str)
)

cluster_df2["local_cluster_id"] = (
    "20210401_" +
    cluster_df2["cluster_id"].astype(str)
)


In [75]:
print(cluster_df["local_cluster_id"].head())
print(cluster_df2["local_cluster_id"].head())


3392590    20210331_0
3392591    20210331_0
3392592    20210331_1
3392593    20210331_2
3392594    20210331_3
Name: local_cluster_id, dtype: object
0    20210401_0
1    20210401_1
2    20210401_2
3    20210401_3
4    20210401_4
Name: local_cluster_id, dtype: object


In [76]:
cross_cluster_edges = [
    (
        f"20210331_{c1}",
        f"20210401_{c2}"
    )
    for c1, c2 in cross_cluster_pairs
]

print(
    "Global cluster edges:",
    len(cross_cluster_edges)
)

print(
    cross_cluster_edges[:20]
)


Global cluster edges: 2066
[('20210331_883', '20210401_7406'), ('20210331_4514', '20210401_4170'), ('20210331_89', '20210401_77'), ('20210331_7669', '20210401_5843'), ('20210331_1050', '20210401_84'), ('20210331_7851', '20210401_4688'), ('20210331_7658', '20210401_92'), ('20210331_178', '20210401_5753'), ('20210331_8240', '20210401_3974'), ('20210331_7997', '20210401_52'), ('20210331_12067', '20210401_384'), ('20210331_7238', '20210401_3526'), ('20210331_7952', '20210401_7219'), ('20210331_422', '20210401_6909'), ('20210331_8024', '20210401_5783'), ('20210331_3861', '20210401_1696'), ('20210331_11482', '20210401_3113'), ('20210331_11660', '20210401_9572'), ('20210331_234', '20210401_5662'), ('20210331_7545', '20210401_2672')]


In [77]:
march_nodes = [
    f"20210331_{i}"
    for i in range(12_641)
]

april_nodes = [
    f"20210401_{i}"
    for i in range(12_901)
]

all_nodes = march_nodes + april_nodes

node_to_index = {
    node: i
    for i, node in enumerate(all_nodes)
}


In [78]:
edge_array = np.array([
    [
        node_to_index[a],
        node_to_index[b]
    ]
    for a, b in cross_cluster_edges
])


In [79]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

n_nodes = len(all_nodes)

rows = edge_array[:, 0]
cols = edge_array[:, 1]

graph = coo_matrix(
    (
        np.ones(len(edge_array), dtype=np.int8),
        (rows, cols)
    ),
    shape=(n_nodes, n_nodes)
)

graph = graph + graph.T


In [80]:
n_global_components, global_labels = connected_components(
    graph,
    directed=False
)

print(
    "Local clusters:",
    n_nodes
)

print(
    "Global connected components:",
    n_global_components
)


Local clusters: 25542
Global connected components: 23476


In [81]:
n_march_clusters = cluster_df["cluster_id"].nunique()
n_april_clusters = cluster_df2["cluster_id"].nunique()

march_nodes = [
    f"20210331_{i}"
    for i in range(n_march_clusters)
]

april_nodes = [
    f"20210401_{i}"
    for i in range(n_april_clusters)
]
